In [1]:
# Estrategia Back Over 2.5

import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("../data_total/dados_betfair.csv", sep=";")

In [13]:
# Filtar colunas para análise
datatest = data[['League', 'Home', 'Away', 'Goals_H_HT', 'Goals_A_HT', 'Goals_H_FT', 'Goals_A_FT', 'Odd_H_Back', 'Odd_A_Back', 'Odd_Over25_FT_Back']].copy()

# Verificar se o jogo teve over 2.5
datatest['WCS'] = np.where(datatest['Goals_H_FT'] + datatest['Goals_A_FT'] > 2.5, 1, 0)

# Definir o valor da aposta
STAKE = 1
COMISSAO = 0.065

# Criar o 'profit' com comissão descontada apenas quando ganha
datatest['Profit'] = round(datatest.apply(
    lambda row: 
        # Se ganhou: ((odd * stake) - stake) * (1 - comissao)
        ((row['Odd_Over25_FT_Back'] * STAKE) - STAKE) * (1 - COMISSAO)
        if row['WCS'] == 1
        # Se perdeu: perde a stake integral (comissão não se aplica)
        else -STAKE, 
    axis=1), 2)

In [14]:
datatest.head()

,League,Home,Away,Goals_H_HT,Goals_A_HT,Goals_H_FT,Goals_A_FT,Odd_H_Back,Odd_A_Back,Odd_Over25_FT_Back,WCS,Profit
0,SPAIN 1,Mallorca,Granada CF,0,0,1,0,1.88,5.30,2.50,0,-1.00
1,SPAIN 1,Osasuna,Real Madrid,1,2,2,4,6.60,1.58,1.87,1,0.81
2,SPAIN 1,Getafe,Girona,1,0,1,0,3.35,2.34,1.95,0,-1.00
3,SPAIN 1,Ath Bilbao,Alaves,2,0,2,0,1.59,7.60,2.28,0,-1.00
4,ENGLAND 1,Fulham,Tottenham,1,0,3,0,3.50,2.10,1.55,1,0.51


In [ ]:
# Função para criar faixas de odds
def criar_faixa_h_back(odd):
    if odd < 1.80:
        return '1.50-1.79'
    elif odd < 2.10:
        return '1.80-2.09'
    elif odd < 2.50:
        return '2.10-2.49'
    elif odd < 3.00:
        return '2.50-2.99'
    elif odd < 3.50:
        return '3.00-3.49'
    elif odd < 4.00:
        return '3.50-3.99'
    elif odd < 5.00:
        return '4.00-4.99'
    else:
        return '5.00+'
    
def criar_faixa_a_back(odd):
    if odd < 1.80:
        return '1.50-1.79'
    elif odd < 2.10:
        return '1.80-2.09'
    elif odd < 2.50:
        return '2.10-2.49'
    elif odd < 3.00:
        return '2.50-2.99'
    elif odd < 3.50:
        return '3.00-3.49'
    elif odd < 4.00:
        return '3.50-3.99'
    elif odd < 5.00:
        return '4.00-4.99'
    else:
        return '5.00+'
    
def criar_faixa_over25_back(odd):
    if odd < 1.80:
        return '1.50-1.79'
    elif odd < 2.10:
        return '1.80-2.09'
    elif odd < 2.50:
        return '2.10-2.49'
    elif odd < 3.00:
        return '2.50-2.99'
    elif odd < 3.50:
        return '3.00-3.49'
    elif odd < 4.00:
        return '3.50-3.99'
    elif odd < 5.00:
        return '4.00-4.99'
    else:
        return '5.00+'
    
# Aplicar as funções às colunas correspondentes
datatest['Faixa_Odd_H_Back'] = datatest['Odd_H_Back'].apply(criar_faixa_h_back)
datatest['Faixa_Odd_A_Back'] = datatest['Odd_A_Back'].apply(criar_faixa_a_back)
datatest['Faixa_Odd_Over25_Back'] = datatest['Odd_A_Back'].apply(criar_faixa_over25_back)

# Agrupars por faixas e calcular estatísticas
print("\n📊 ANÁLISE POR FAIXAS (recomendado para amostras maiores):")
print("-" * 80)

agrupando_faixas = datatest.groupby(['Faixa_Odd_H_Back', 'Faixa_Odd_A_Back', 'Faixa_Odd_Over25_Back']).agg(
    Total_Jogos=('WCS', 'count'),
    Jogos_0x1=('WCS', 'sum'),
    Percentual_Acerto=('WCS', lambda x: (x.sum() / len(x) * 100)),
    Lucro_Total=('Profit', 'sum')
).reset_index()

# Arredondar valores
agrupando_faixas['Percentual_Acerto'] = agrupando_faixas['Percentual_Acerto'].round(2)

# Odd justa
agrupando_faixas['Odd_Justa'] = round(1 / (agrupando_faixas['Percentual_Acerto'] / 100), 2)

# Agrupar por lucro Total
agrupando_faixas = agrupando_faixas.sort_values(by='Lucro_Total', ascending=False)


agrupando_faixas.head(3)